<div class="title-wrap">
  <h1 class="title-main" style="font-weight: bold; font-size: 2.65rem; margin-bottom: 0.5rem;">
  Waterloo-Park-LiDAR-Tree-Detection-Pipeline
</h1>
<h2 class="title-sub" style="font-style: italic; font-size: 1.8rem; margin-top: 0rem; margin-bottom: 0.2rem;">
  A Machine Learning Exploration of LiDAR Classification
</h2>
</div>

# Module 1: *Data Exploration*
##### Version Number: 1.0
---
### Contents  
---
### Notes
---
### Inputs
---
### Outputs  
---
### User Created Dependencies  
---
### Third Party Dependencies

In [ ]:
import pandas
import laspy
from osgeo import gdal, osr

### Load LAS file

In [ ]:
# Load the LAS/LAZ file
las = laspy.read("data/clipped_point_cloud.las")

# Build a DataFrame with only the fields you want
cloud = pd.DataFrame({
    "x": las.x,
    "y": las.y,
    "z": las.z,
    "classification": las.classification
    "keypoint": las.keypoint_flag
})

ds = gdal.Open("data/clipped_ortho.tif")

### Load Orthophoto Raster

In [ ]:
## function to get basic raster information
gt = ds.GetGeoTransform()

x_min = gt[0]          # upper-left X
d     = gt[1]          # pixel width
y_max = gt[3]          # upper-left Y
pixel_height = gt[5]   # negative

cols = ds.RasterXSize    # number of columns
ny = ds.RasterYSize    # number of rows

x_max = x_min + nx * d
y_min = y_max + ny * pixel_height   # pixel_height is negative

### Find LiDAR points within each Grid

In [ ]:
# Column index (how far right the point is)
i = ((cloud.x - x_min) / d).astype(int)

# Row index (how far down the point is — note the Y flip)
j = ((y_max - cloud.y) / d).astype(int)

# Keep only points that fall inside the raster extent
valid = (
    (i >= 0) & (i < cols) &
    (j >= 0) & (j < rows)
)

# Filter indices and point objects
i_valid = i[valid]
j_valid = j[valid]
points_valid = cloud[valid]   # actual point references

# (Optional) Build a density raster for visualization
density = np.zeros((rows, cols), dtype=int)

# Increment the count for each valid point's cell
np.add.at(density, (j_valid, i_valid), 1)

# Store ACTUAL point objects into bins
# Loop over the filtered points and place each one
# into its corresponding raster cell.

for idx in range(len(points_valid)):
    row = j_valid[idx]
    col = i_valid[idx]
    bins[row][col].append(points_valid[idx])


### Output Density Raster

In [ ]:

# Create a new GeoTIFF with the same spatial reference

driver = gdal.GetDriverByName("GTiff")
out_ds = driver.Create(
    "density.tif",
    cols,          # width
    rows,          # height
    1,             # number of bands
    gdal.GDT_Int32 # data type
)

# Apply the same geotransform as the orthophoto
out_ds.SetGeoTransform(gt)

# Apply the same projection as the orthophoto
out_ds.SetProjection(ds.GetProjection())


# Write the density raster to band 1


out_band = out_ds.GetRasterBand(1)
out_band.WriteArray(density)

# Optional: set NoData value
out_band.SetNoDataValue(-9999)

# Flush to disk
out_band.FlushCache()
out_ds = None